[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q1/03_consensus_compare.ipynb)

# R1-Q1 Week 4 — Consensus Comparison & Report

You're in Week 4 of R1-Q1. Last week you identified the common stress core — 9,067 genes differentially expressed in at least 6 of the 8 abiotic stresses in AtGenExpress — and ran GO over-representation analysis to characterize what those genes do functionally. This week takes that core and compares it to a published meta-analytic consensus: Hakkak & Tohidfar 2026's signature of conserved drought and salt response in *Arabidopsis thaliana*. The comparison closes the R1-Q1 arc and produces the material for the final paper.

**By the end of this notebook you will have:**

- The core gene set translated from ATH1 probe IDs to AGI locus codes.
- A pre-committed prediction of how the two cores should overlap, written down before seeing the data.
- The overlap result — shared genes, ours-only, theirs-only — with hypergeometric significance, plus a three-region breakdown checked against the prediction.
- Targeted comparisons against Hakkak & Tohidfar's transcription factor list and the specific hub genes named in their paper.
- A synthesis pulling the comparison together with the enrichment findings from Notebook 2, and a saved comparison artifact for the final paper.

## 1) Setup and data

Install the iRI Lab library, mount Drive, read the common stress core that Notebook 2 wrote to Drive, and sanity-check what loaded.

In [ ]:
# Setup. Four steps: install the iRI Lab library, mount Drive,
# read the common stress core that last week's notebook wrote to Drive,
# and sanity-check what loaded.

# 1. Install the library (not pre-installed on Colab).
!pip install --upgrade --no-cache-dir git+https://github.com/geraldmc/irilab2026.git@main --quiet

In [ ]:
# Setup Step 2:  Mount Drive, read the common stress core, and sanity-check what loaded.

# 2. Mount Drive.
import irilab2026 as iri
iri.setup()

# 3. Read the common stress core from Drive — written by 02_core_overlap.ipynb.
import pandas as pd
core_genes = pd.read_parquet(iri.output_dir("r1_q1") / "core_genes.parquet")

# 4. Sanity-check what loaded.
print(f"Loaded: {core_genes.shape}")
print(f"n_stresses distribution:\n{core_genes['n_stresses'].value_counts().sort_index()}")
core_genes.head()

### 1.1) Inputs this notebook needs

Two files:

1. **`core_genes.parquet`** — produced by Notebook 2, sitting in your Drive at `irilab2026_outputs/r1_q1/`. The cell above confirms it is accessible.

2. **Hakkak & Tohidfar 2026 Supplementary File 1** — the meta-analytic DEG list from the published paper. We load this in Section 4. Download instructions and the file naming wrinkle are in that section.

## 2) Translate probes to AGI

The core we just loaded is keyed on ATH1 probe IDs (e.g., `244903_at`). Hakkak & Tohidfar's consensus list is keyed on AGI codes (e.g., `At5g59310`) — the standard *Arabidopsis* Genome Initiative identifiers used in nearly every gene-level resource. We can't compare the two lists directly until we translate one of them. We translate ours.

We also need a **background** — the full set of genes that *could* have ended up in the core. The background is every probe that appeared in last week's DE analysis, whether it was flagged DE or not. We need it because Section 4 asks "is the overlap with Hakkak's list bigger than chance?", and answering that requires knowing the universe the core was drawn from. We translate the background to AGI too, so the comparison runs on a common identifier system.

The translation uses the GPL198 annotation table — GEO's accession for the Affymetrix ATH1 platform, which maps every probe ID on the chip to its target AGI locus. Two defensive cases come up. **Multi-locus probes** target more than one AGI locus, listed as `At1g01010 /// At1g01020`; we take the first locus, which is the probe's primary target. **Probes without AGI** can't participate in the comparison and get dropped — this is also where the *Affymetrix spike-in control probes* you saw in the core (prefixed `AFFX-`, bacterial sequences used for hybridization quality control) drop out, since they aren't *Arabidopsis* genes and have no AGI codes.

In [ ]:
# Load the long-format DE table from Notebook 2. We need every probe that
# appeared in the DE analysis — that's the universe for the hypergeometric
# test in Section 4.
de_results = pd.read_parquet(iri.output_dir("r1_q1") / "de_results.parquet")

print(f"DE results loaded: {de_results.shape}")
print(f"Unique probes:     {de_results['gene'].nunique():,}")

In [ ]:
# Fetch the GPL198 annotation table. GEOparse caches the SOFT file in
# iri.cache_dir() so a re-run doesn't hit the network a second time.
import GEOparse

gpl = GEOparse.get_GEO(geo='GPL198', destdir=str(iri.cache_dir()), silent=True)

print(f"Annotation table: {gpl.table.shape}")
gpl.table[['ID', 'AGI']].head()

In [ ]:
# Build a probe -> AGI mapping from the GPL198 annotation table.
#
# Two cases to handle defensively:
#   1. Multi-locus probes: a probe may target multiple AGI loci, separated
#      by ' /// '. We take the first locus (the probe's primary target).
#   2. Missing AGI: some probes have no AGI annotation at all (empty or NaN).
#      Those probes can't participate in the comparison and get dropped.
#      This is where AFFX control probes get filtered out.
probe_to_agi = {}
for _, row in gpl.table[['ID', 'AGI']].dropna().iterrows():
    probe = row['ID']
    first_locus = str(row['AGI']).split(' /// ')[0].strip().upper()
    if first_locus:
        probe_to_agi[probe] = first_locus

print(f"Probes with AGI mapping:  {len(probe_to_agi):,}")
print(f"Probes without AGI:       {len(gpl.table) - len(probe_to_agi):,}")

# Translate the core (9,067 probes -> AGI codes).
core_agi = (
    core_genes['gene']
    .map(probe_to_agi)
    .dropna()
    .unique()
    .tolist()
)

# Translate the background — every probe we ran DE on.
background_agi = (
    de_results['gene']
    .drop_duplicates()
    .map(probe_to_agi)
    .dropna()
    .unique()
    .tolist()
)

print()
print(f"Core (AGI):       {len(core_agi):,} unique loci")
print(f"Background (AGI): {len(background_agi):,} unique loci")

## 3) Pre-committed prediction

Before loading Hakkak & Tohidfar's data, we write down what we expect the comparison to show. Same pattern as Notebook 2's N=6 supermajority commitment: decide the rules in advance, in writing, so the result doesn't reduce to "whichever interpretation makes the answer look cleanest."

A comparison between two gene lists has three regions. **The intersection** — genes in both lists. **Theirs-only** — genes in Hakkak's consensus that aren't in our core. **Ours-only** — genes in our core that aren't in Hakkak's consensus. Each region gets its own prediction about what biology should dominate it.

The framing matters because the comparison isn't apples-to-apples. Our core spans eight abiotic stresses and counts a gene if it's DE in at least six. Hakkak's consensus is built from a meta-analysis of drought-and-salt-only studies. The overlap should be the cross-tolerant subset of drought/salt response by construction — genes that respond not just to drought and salt but to most of our other six stresses as well. The non-overlapping regions should each pick up biology specific to the broader-versus-narrower stress set.

The predictions, written down before we look:

- **Intersection.** Shared cross-tolerance machinery — broad ROS (reactive oxygen species) metabolism, generic stress-responsive TFs like WRKY and NAC, HSPs (heat shock proteins) that act across multiple proteostasis stresses. The backbone of stress response, not drought/salt-specific biology.
- **Theirs-only.** Drought/salt-specific biology — ion homeostasis (the SOS pathway, Na⁺/K⁺ transporters), water-deprivation specifics (LEA proteins, dehydrins), osmolyte biosynthesis, ABA-specific signaling. Genes whose response is concentrated in water/ion stress rather than spread across many stresses.
- **Ours-only.** Biology shared across our eight stresses but not specifically drought/salt — hypoxia/decreased-oxygen response (Notebook 2's top enrichment cluster), and wound/genotoxic/UV-specific machinery that recurs across our other stresses.

A result that would genuinely surprise us: if Hakkak's drought/salt-specific genes (ion transporters, LEA proteins, ABA-specific components) appear substantially in our core. That would contradict the textbook view that ion homeostasis is salt-specific and water-deprivation machinery is drought-specific. We note this as the falsification anchor — if it shows up, the prediction was wrong in a substantive way and Section 7's synthesis has to account for it.

The cell below captures the prediction as a Python dict so Section 4's three-region breakdown can reference it explicitly. Pre-committed means pre-committed — the dict gets written once, here.

In [ ]:
# The three-region prediction, captured as a dict so Section 4 can
# reference it explicitly rather than re-stating it. Pre-committed
# means committed in writing before we look at Hakkak's data.
prediction = {
    "intersection": (
        "Shared cross-tolerance machinery — broad ROS metabolism, "
        "generic stress-responsive TFs (WRKY, NAC), HSPs."
    ),
    "theirs_only": (
        "Drought/salt-specific biology — ion homeostasis (SOS pathway, "
        "Na+/K+ transporters), LEA proteins, dehydrins, osmolyte "
        "biosynthesis, ABA-specific signaling."
    ),
    "ours_only": (
        "Cross-stress but not drought/salt-specific — hypoxia response "
        "(Notebook 2's top enrichment cluster), and wound/genotoxic/UV "
        "machinery that recurs across our other stresses."
    ),
    "would_surprise": (
        "Drought/salt-specific genes appearing substantially in our core — "
        "would contradict the canonical view that ion homeostasis is "
        "salt-specific and water-deprivation machinery is drought-specific."
    ),
}

for region, text in prediction.items():
    label = region.upper().replace("_", " ")
    print(f"\n{label}:")
    print(f"  {text}")

### 3.1) Load Hakkak & Tohidfar's consensus

The CSV file at `irilab2026_outputs/r1_q1/hakkak_2026_supp1.csv` (on Google Drive) holds the differentially expressed genes that Hakkak & Tohidfar identified through their meta-analysis — 397 up-regulated under drought-or-salt and 170 down-regulated. This is the list that we will compare our core to in Section 4.

The file has an unusual layout. Row 1 is a title. Row 2 is the column header. Below the header, the data is split into two side-by-side blocks: columns 1–8 hold the up-regulated genes, then two empty separator columns, then columns 11–18 hold the down-regulated genes. The blocks have different lengths (397 rows vs. 170 rows), so the down-regulated side runs out partway down with empty cells. We need to read both blocks and stack them into a single long-format DataFrame.

One acknowledgment before we start. The paper's abstract and methods both report **576** total DEGs. The supplementary file contains **567** (397 + 170). The 9-gene difference isn't explained in the paper text — likely a transcription discrepancy between manuscript and supplementary that didn't get caught in proofing. The supplementary is the authoritative source, since it's the file the comparison actually uses. We work with 567.

To make sure the file loaded correctly, we validate against two anchors that the paper text names by gene: **At5g59310 (LTP4)** should be the top up-regulated gene at log2FC ≈ 4.46, and **At1g22690 (GASA9)** should be the top down-regulated gene at log2FC ≈ -2.56. If both check out, the file is what the paper says it is and we proceed.

In [ ]:
# Load Hakkak & Tohidfar's primary DEG list. The CSV has an unusual
# layout: title row, header row, then two side-by-side blocks of 8
# columns separated by two empty columns. Up-regulated genes on the
# left, down-regulated on the right.
hakkak_path = iri.output_dir("r1_q1") / "hakkak_2026_supp1.csv"
try:
    raw = pd.read_csv(hakkak_path, header=1)
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find Hakkak & Tohidfar's supplementary file 1.\n"
        f"  Expected at: {hakkak_path}\n\n"
        f"In Google Drive, upload `hakkak_2026_supp1.csv` to:\n"
        f"  MyDrive/irilab2026_outputs/r1_q1/\n"
        f"then re-run this cell."
    ) from None

# Split into the two parallel blocks and rename to consistent columns.
cols = ["Gene", "Up_Down", "log2FC", "AveExpr", "t", "P_Value", "adj_P_Val", "B"]
up = raw.iloc[:, 0:8].copy()
down = raw.iloc[:, 10:18].copy()
up.columns = cols
down.columns = cols

# The down-regulated block is shorter (170 vs 397), so drop empty rows.
up = up.dropna(subset=["Gene"]).reset_index(drop=True)
down = down.dropna(subset=["Gene"]).reset_index(drop=True)

# Stack the two blocks and uppercase AGI codes to match Notebook 2's
# probe_to_agi convention (which uppercased on the way in).
hakkak = pd.concat([up, down], ignore_index=True)
hakkak["Gene"] = hakkak["Gene"].str.upper()

print(f"Up-regulated:   {len(up)}")
print(f"Down-regulated: {len(down)}")
print(f"Total unique:   {hakkak['Gene'].nunique()}")
hakkak.head()

In [ ]:
# Validate against the anchors named in the paper text:
#   At5g59310 (LTP4)  — top up-regulated, log2FC ≈ 4.46
#   At1g22690 (GASA9) — top down-regulated, log2FC ≈ -2.56
top_up = up.sort_values("log2FC", ascending=False).iloc[0]
top_down = down.sort_values("log2FC", ascending=True).iloc[0]

# Note: 'up' and 'down' here are still in their original case (At...);
# we only uppercased after concatenation. Uppercase for the comparison.
top_up_gene = top_up['Gene'].upper()
top_down_gene = top_down['Gene'].upper()

print(f"Top up-regulated:   {top_up_gene:12s} log2FC={top_up['log2FC']:.3f}  (paper: AT5G59310, 4.46)")
print(f"Top down-regulated: {top_down_gene:12s} log2FC={top_down['log2FC']:.3f}  (paper: AT1G22690, -2.56)")

assert top_up_gene == 'AT5G59310', f"Expected AT5G59310 as top up, got {top_up_gene}"
assert top_down_gene == 'AT1G22690', f"Expected AT1G22690 as top down, got {top_down_gene}"
print("\nAnchors check out.")

# The list of AGI codes that Section 4 will compare against our core_agi.
hakkak_agi = hakkak["Gene"].unique().tolist()
print(f"\nhakkak_agi: {len(hakkak_agi):,} unique AGI codes")

## 4) Overlap analysis and three-region breakdown

This is the methodological centerpiece of the notebook. We take our core (translated to AGI in Section 1) and Hakkak & Tohidfar's consensus (loaded in Section 3), compute the three regions of overlap, and check the results against the prediction Section 2 pre-committed to.

Three numbers do most of the work. **The intersection** is the count of AGI codes that appear in both lists. **Theirs-only** is the count in Hakkak's list that aren't in our core. **Ours-only** is the count in our core that aren't in Hakkak's list. The three numbers, plus a total, fully describe how the lists relate.

To check whether the overlap is bigger than chance, we run a **hypergeometric test**. The intuition: imagine the background as an urn full of marbles, with Hakkak's genes colored red and the rest colored blue. We draw a number of marbles equal to our core size and count how many are red. The hypergeometric distribution tells us the probability of getting at least that many reds by random draw. A very small p-value means the overlap is much larger than chance — biology, not coincidence.

We use the full `background_agi` (~22,000 testable AGI codes) as the universe. A more conservative version of the test would restrict the universe to AGI codes detectable on both Hakkak's and our platforms; that would shift the p-value modestly but not change the headline finding when the overlap is far from random.

The section ends with a side-by-side display of each region's count and the pre-committed prediction for what biology it should contain. The interpretation — did the prediction hold, what biology actually dominates each region — is the work of Sections 5 through 7. This section produces the numbers and leaves the reading for later.

### 4.1) Computing the three regions

Three set operations give us the gene counts for each Venn region. We convert each list to a Python set — sets dedupe automatically and support overlap arithmetic with the `&` (intersection) and `-` (difference) operators — then compute the intersection, theirs-only, and ours-only. The percentages at the bottom tell us recovery in each direction: what fraction of Hakkak's list ended up in our core, and what fraction of our core is in Hakkak's consensus.

In [ ]:
# Compute the three regions. Sets are the right data structure for
# overlap arithmetic — order doesn't matter, dedup is automatic.
core_set       = set(core_agi)
hakkak_set     = set(hakkak_agi)
background_set = set(background_agi)

intersection = core_set & hakkak_set
ours_only    = core_set - hakkak_set
theirs_only  = hakkak_set - core_set

n_intersection = len(intersection)
n_ours_only    = len(ours_only)
n_theirs_only  = len(theirs_only)

print(f"Our core:           {len(core_set):,} AGI codes")
print(f"Hakkak consensus:   {len(hakkak_set):,} AGI codes")
print(f"Background (universe): {len(background_set):,} AGI codes")
print()
print(f"Intersection:       {n_intersection:,} genes")
print(f"Theirs-only:        {n_theirs_only:,} genes")
print(f"Ours-only:          {n_ours_only:,} genes")
print()
print(f"Of Hakkak's {len(hakkak_set)} genes, {n_intersection} ({100*n_intersection/len(hakkak_set):.1f}%) are in our core.")
print(f"Of our {len(core_set)} core genes, {n_intersection} ({100*n_intersection/len(core_set):.1f}%) are in Hakkak's consensus.")

### 4.2) Hypergeometric significance test

The hypergeometric test answers a single question: given a universe of `N` genes, of which `K` are in Hakkak's list, if we draw a sample of `n` (our core), how likely are we to see at least `k` of them in Hakkak's list by chance alone? A very small p-value means the observed overlap is far larger than chance — biology rather than coincidence.

We use the full `background_agi` as the universe. A more conservative version of the test would restrict the universe to AGI codes detectable on both Hakkak's and our platforms; that would shift the p-value modestly but not change the headline finding when the overlap is far from random.

In [ ]:
# Hypergeometric test: is the overlap larger than expected by chance?
#
# Setup:
#   N = background size (universe of testable AGI codes)
#   K = Hakkak genes that are also in our background (the "successes" present in the universe)
#   n = our core size (the sample drawn)
#   k = observed overlap
from scipy.stats import hypergeom

N = len(background_set)
K = len(hakkak_set & background_set)
n = len(core_set)
k = n_intersection

# Expected overlap under random draw = n * K / N
expected = n * K / N

# P(X >= k) — survival function at k-1
p_value = hypergeom.sf(k - 1, N, K, n)

print(f"Background size (N):             {N:,}")
print(f"Hakkak genes in background (K):  {K:,}")
print(f"Our core size (n):               {n:,}")
print(f"Observed overlap (k):            {k:,}")
print()
print(f"Expected overlap under chance:   {expected:.1f}")
print(f"Fold enrichment over expected:   {k / expected:.2f}×")
print(f"Hypergeometric p-value:          {p_value:.3e}")

### 4.3) Visualizing the overlap

A two-set Venn diagram is the right visual here — three regions, three numbers. The `matplotlib_venn` package handles the proportional circles automatically. We install it at point of use rather than bundling it in `irilab2026`, since only this notebook needs it.

The asymmetry of the diagram will be the main thing worth noticing: Hakkak's small circle should be almost entirely contained within our larger one, with the recovery percentage from 4.1 reflected in how much of their circle gets eaten by the intersection.

In [ ]:
# Render a 2-set Venn diagram. matplotlib_venn isn't bundled with the
# library because only this notebook needs it — install at point of use.
!pip install matplotlib_venn --quiet

from matplotlib_venn import venn2
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))
venn2(
    subsets=(n_ours_only, n_theirs_only, n_intersection),
    set_labels=("Our core\n(8 stresses)", "Hakkak & Tohidfar\n(drought + salt)"),
    ax=ax,
)
ax.set_title(
    f"AtGenExpress single-dataset core vs. Hakkak & Tohidfar 2026 consensus\n"
    f"Hypergeometric p = {p_value:.2e}, {k / expected:.1f}× enrichment over chance"
)
plt.tight_layout()
plt.show()

### 4.4) Comparing the result to the pre-committed prediction

The three regions have their counts. The pre-committed prediction from Section 3 said what kind of biology should dominate each region. Below we print them side-by-side — region count on top, predicted biology below — plus the falsification anchor at the end.

This section produces the numbers and the prediction text together. Whether the prediction actually *held* — whether the genes in each region are biologically what we said they should be — is the work of Sections 5, 6, and 7. Section 5 looks at transcription factors specifically. Section 6 spot-checks the hub genes Hakkak named by name. Section 7 synthesizes.

In [ ]:
# Side-by-side: each region's count next to the pre-committed prediction.
# The prediction dict was set in Section 2 and hasn't been touched since.
results = {
    "intersection": n_intersection,
    "theirs_only":  n_theirs_only,
    "ours_only":    n_ours_only,
}

for region, count in results.items():
    label = region.upper().replace("_", " ")
    print(f"\n{label} — {count:,} genes")
    print(f"Predicted: {prediction[region]}")

print(f"\n{'-' * 60}")
print("FALSIFICATION ANCHOR")
print(f"Predicted: {prediction['would_surprise']}")
print("\nWhether this happened is the question Section 6 and Section 7 answer.")

## 5) Transcription factor sub-analysis

Notebook 2's enrichment found no canonical stress-response TF families in the top 20 BP terms. That checked which GO categories are over-represented in our core overall — it didn't check whether specific TF genes are in our core. This section asks the narrower, more targeted question: of the 60 transcription factors Hakkak & Tohidfar flagged as differentially expressed in drought and salt, how many are in our core, and how does the recovery break down by family?

The R1-Q1 prediction named AP2/ERF, MYB, WRKY, NAC, and bZIP as the canonical stress-response TF families that should appear in any common stress core. Hakkak's analysis identified TFs in 15 families across their 567 DEGs, with NAC, ERF, MYB, bHLH, and WRKY as the most populous. If our core is a genuine stress core, those families should be well-represented in the intersection — even though they didn't surface as enriched *terms* in Notebook 2's GO analysis.

The output is a recovery pattern, family by family. Section 7's synthesis reads it.

### 5.1) Load Hakkak's TF table

The TF table (Supplementary File 3, `hakkak_2026_supp3.csv`) uses the same side-by-side CSV layout as the consensus file in Section 3: title row, header row, then up-regulated TFs in the left block (columns 1–4: ORF, Symbol, Family, Group) and down-regulated TFs in the right block (columns 6–9), separated by one empty column.

We parse the layout the same way as Section 3.1, stack the two blocks into one long-format DataFrame, and uppercase the AGI codes (column `ORF`) to match the convention used for `core_agi`. The result is `hakkak_tf`, with 60 rows and four columns. We confirm the load by checking the family distribution against the paper: NAC, ERF, MYB, bHLH, and WRKY should be the top five families.

In [ ]:
# Load Hakkak's TF table. Same side-by-side CSV layout as the consensus
# file in Section 3 — title row, header row, then up-regulated TFs in
# columns 1-4 (left block) and down-regulated TFs in columns 6-9 (right
# block), separated by one empty column.
tf_path = iri.output_dir("r1_q1") / "hakkak_2026_supp3.csv"
raw_tf = pd.read_csv(tf_path, header=1)

# Split the two blocks.
tf_cols = ["ORF", "Symbol", "Family", "Group"]
up_tf = raw_tf.iloc[:, 0:4].copy()
down_tf = raw_tf.iloc[:, 5:9].copy()
up_tf.columns = tf_cols
down_tf.columns = tf_cols

# Drop empty rows in each block.
up_tf = up_tf.dropna(subset=["ORF"]).reset_index(drop=True)
down_tf = down_tf.dropna(subset=["ORF"]).reset_index(drop=True)

# Stack and uppercase AGI codes to match core_agi.
hakkak_tf = pd.concat([up_tf, down_tf], ignore_index=True)
hakkak_tf["ORF"] = hakkak_tf["ORF"].str.upper()

print(f"Up-regulated TFs:   {len(up_tf)}")
print(f"Down-regulated TFs: {len(down_tf)}")
print(f"Total TFs:          {len(hakkak_tf)}")
print(f"Families:           {hakkak_tf['Family'].nunique()}")
print()
print("Family distribution:")
print(hakkak_tf['Family'].value_counts())

### 5.2) Cross-reference with our core

A single set operation: which of Hakkak's 60 TFs are in our `core_set`? We add an `in_core` boolean column to the TF table so the family-level breakdown in 5.3 can use it directly.

The interesting payoff is being able to see the named genes for the first time in this notebook. The R1-Q1 prediction mentioned WRKY, NAC, and other families generically; here we can see *which specific TFs* (DREB2A, NAC019, ATHB-7, etc.) made it into the intersection and which didn't.

In [ ]:
# Cross-reference Hakkak's 60 TFs with our core.
hakkak_tf["in_core"] = hakkak_tf["ORF"].isin(core_set)

n_in_core = hakkak_tf["in_core"].sum()
n_total = len(hakkak_tf)

print(f"Of Hakkak's {n_total} TFs, {n_in_core} ({100*n_in_core/n_total:.0f}%) are in our core.")
print()
print("TFs in our core (sorted by family, then symbol):")
in_core_tfs = hakkak_tf[hakkak_tf["in_core"]].sort_values(["Family", "Symbol"])
print(in_core_tfs[["ORF", "Symbol", "Family", "Group"]].to_string(index=False))
print()
print("TFs NOT in our core:")
not_in_core_tfs = hakkak_tf[~hakkak_tf["in_core"]].sort_values(["Family", "Symbol"])
print(not_in_core_tfs[["ORF", "Symbol", "Family", "Group"]].to_string(index=False))

### 5.3) Family-level breakdown

The family-level question: are recovery rates roughly uniform across families, or is there an asymmetry — some families well-represented in our core, others under-represented?

The summary table below shows each family's total TF count, how many are in our core, and the recovery percentage. The stacked bar chart visualizes the same information — for each family, the steel-blue portion is "in our core" and the gray portion is "not in our core."

A family that's well-recovered (mostly blue) means TFs from that family respond broadly across our eight stresses, not just drought and salt. A family that's poorly recovered (mostly gray) means TFs from that family responded in Hakkak's drought-and-salt analysis but didn't carry through to our broader stress core — suggesting they're drought/salt-specific rather than broadly stress-responsive.

In [ ]:
# Family-level breakdown: how does recovery vary across TF families?
family_summary = (
    hakkak_tf
    .groupby("Family")
    .agg(total=("ORF", "count"), in_core=("in_core", "sum"))
    .reset_index()
)
family_summary["recovery_pct"] = (100 * family_summary["in_core"] / family_summary["total"]).round(1)
family_summary = family_summary.sort_values("total", ascending=False).reset_index(drop=True)

print(family_summary.to_string(index=False))
print()

# Stacked bar chart: total TFs per family, with "in our core" portion
# colored. Visually obvious which families are mostly recovered.
fig, ax = plt.subplots(figsize=(10, 5))
families = family_summary["Family"]
in_core = family_summary["in_core"]
not_in_core = family_summary["total"] - in_core

x = range(len(families))
ax.bar(x, in_core, label="In our core", color="steelblue")
ax.bar(x, not_in_core, bottom=in_core, label="Not in our core", color="lightgray")
ax.set_xticks(x)
ax.set_xticklabels(families, rotation=45, ha="right")
ax.set_ylabel("Number of TFs in Hakkak's list")
ax.set_title("Hakkak & Tohidfar's TFs by family — how many are in our core")
ax.legend()
plt.tight_layout()
plt.show()

## 6) Hub-gene spot-check

Section 5 looked at families. This section looks at specific named genes — the genes Hakkak called out by name in their paper text, and the classical drought/salt markers from the textbook canon. Two questions, one per subsection.

The pre-committed falsification anchor from Section 3 was that drought/salt-*specific* effectors — ion homeostasis, LEA proteins, dehydrins — should remain in theirs-only and not appear in our core. Section 5 showed that the regulatory machinery (TFs) is broadly cross-stress; this section asks whether the *effector* machinery also is, or whether the prediction holds for the effectors specifically.

### 6.1) Genes named in Hakkak's discussion

Hakkak named about 22 specific genes across their results and discussion sections: the two extremes (LTP4 and GASA9), the 10 PPI hub transcription factors from their network analysis, and the 10 high-confidence hub genes from the three-method intersection (which includes the three genes they validated against independent RNA-seq). We check each one against our core.

Most of these are regulatory (TFs and hub genes), so based on Section 5 the recovery rate should be high. The more interesting cases are the candidate hubs that *aren't* TFs — PS2, MEE3, PSK2, TBL45, UGT73C1, MHX — these are downstream effectors and signaling components rather than transcriptional regulators.

In [ ]:
# Specific genes Hakkak named in their paper text — checking each
# against our core. The list combines: the two extreme-response anchors,
# the 10 PPI hub TFs (degree centrality from network analysis), and the
# 10 high-confidence hub genes from the three-method intersection
# (which includes the 3 final validated hubs).
named_genes = [
    # The extremes
    ("LTP4",      "AT5G59310", "Most induced (log2FC=4.46)"),
    ("GASA9",     "AT1G22690", "Most repressed (log2FC=-2.56)"),
    # PPI hub TFs from network analysis
    ("DREB2A",    "AT5G05410", "PPI hub TF (degree 32)"),
    ("AtMYB2",    "AT2G47190", "PPI hub TF (degree 16)"),
    ("ATHB-7",    "AT2G46680", "PPI hub TF (degree 15)"),
    ("NAC019",    "AT1G52890", "PPI hub TF (degree 15)"),
    ("NAC072",    "AT4G27410", "PPI hub TF (degree 13)"),
    ("AZF2",      "AT3G19580", "PPI hub TF (degree 12)"),
    ("ATHB-12",   "AT3G61890", "PPI hub TF (degree 11)"),
    ("NAC002",    "AT1G01720", "PPI hub TF (degree 11)"),
    ("ERF109",    "AT4G34410", "PPI hub TF (degree 10)"),
    ("DREB2B",    "AT3G11020", "PPI hub TF (degree 10)"),
    # Three-method intersection (10 high-confidence candidates)
    ("PS2",       "AT1G73010", "Candidate hub (3-method intersection)"),
    ("NAC032",    "AT1G77450", "Candidate hub (3-method intersection)"),
    ("MEE3",      "AT2G21650", "Candidate hub (3-method intersection)"),
    ("PSK2",      "AT2G22860", "Candidate hub (3-method intersection)"),
    ("WRKY25",    "AT2G30250", "Final validated hub (3-method + RNA-seq)"),
    ("TBL45",     "AT2G30010", "Final validated hub (3-method + RNA-seq)"),
    ("T23K23.24", "AT1G67910", "Candidate hub (3-method intersection)"),
    ("T4C15.27",  "AT2G35070", "Final validated hub (3-method + RNA-seq)"),
    ("UGT73C1",   "AT2G36750", "Candidate hub (3-method intersection)"),
    ("MHX",       "AT2G47600", "Candidate hub (3-method intersection)"),
]

named_df = pd.DataFrame(named_genes, columns=["Symbol", "AGI", "Note"])
named_df["in_core"] = named_df["AGI"].isin(core_set)
named_df["in_hakkak"] = named_df["AGI"].isin(hakkak_set)

n_in_core = named_df["in_core"].sum()
n_total = len(named_df)
print(f"Of {n_total} genes Hakkak named, {n_in_core} ({100*n_in_core/n_total:.0f}%) are in our core.\n")
print(named_df.to_string(index=False))

### 6.2) Canonical drought/salt markers — the falsification anchor

This is the falsification anchor in action. We pick a small set of classical drought/salt-specific marker genes from the textbook canon — RD29A and RD29B (the dehydration-responsive markers, downstream of DREB2A), ERD10, COR47, and LEA14 (LEA family proteins, water-deprivation specific), and SOS1/SOS2/SOS3/HKT1/NHX1 (the salt overly sensitive pathway and the canonical salt-specific transporters). Each gets checked against our background (was it testable on our chip?), Hakkak's consensus (did it pass their drought+salt analysis?), and our core (did it pass our 6-of-8-stresses cutoff?).

The pre-committed prediction said these drought/salt-specific markers should stay in theirs-only. If most of them appear in the intersection (in both Hakkak's consensus AND our core), the prediction was wrong in a substantive way and Section 7 has to account for it. If they're mostly in theirs-only or aren't in Hakkak's list at all, the prediction holds.

The list below isn't exhaustive — it's a focused spot-check using the most classical markers. The AGI codes should be sanity-checked against TAIR if anything looks off; I drafted them from memory and there's room for one to be wrong.

In [ ]:
# Canonical drought/salt-specific markers — the falsification anchor.
# If these classical drought/salt-specific genes appear in our core,
# the prediction from Section 3 was wrong in a substantive way; if they
# stay in theirs-only or aren't in Hakkak's list at all, the prediction
# holds.
canonical_markers = [
    # Drought / water-deprivation specifics
    ("RD29A",  "AT5G52310", "Drought (responsive to desiccation 29A)"),
    ("RD29B",  "AT5G52300", "Drought (responsive to desiccation 29B)"),
    ("ERD10",  "AT1G20450", "Drought (LEA family, early dehydration)"),
    ("COR47",  "AT1G20440", "Drought (LEA family, cold-responsive 47)"),
    ("LEA14",  "AT1G01470", "Drought (Late Embryogenesis Abundant 14)"),
    # Salt-specific
    ("SOS1",   "AT2G01980", "Salt (Na+/H+ antiporter, plasma membrane)"),
    ("SOS2",   "AT5G35410", "Salt (CIPK24 kinase, SOS pathway)"),
    ("SOS3",   "AT5G24270", "Salt (CBL4 calcium sensor, SOS pathway)"),
    ("HKT1",   "AT4G10310", "Salt (high-affinity K+/Na+ transporter)"),
    ("NHX1",   "AT5G27150", "Salt (vacuolar Na+/H+ exchanger)"),
]

markers_df = pd.DataFrame(canonical_markers, columns=["Symbol", "AGI", "Category"])
markers_df["on_chip"]   = markers_df["AGI"].isin(background_set)
markers_df["in_hakkak"] = markers_df["AGI"].isin(hakkak_set)
markers_df["in_core"]   = markers_df["AGI"].isin(core_set)

print(markers_df.to_string(index=False))
print()

n_total       = len(markers_df)
n_on_chip     = markers_df["on_chip"].sum()
n_in_hakkak   = markers_df["in_hakkak"].sum()
n_in_core     = markers_df["in_core"].sum()
n_intersection = ((markers_df["in_hakkak"]) & (markers_df["in_core"])).sum()
n_theirs_only = ((markers_df["in_hakkak"]) & (~markers_df["in_core"])).sum()

print(f"Out of {n_total} canonical drought/salt markers:")
print(f"  {n_on_chip} are on the ATH1 chip (could have been DE in our analysis)")
print(f"  {n_in_hakkak} are in Hakkak's consensus")
print(f"  {n_in_core} are in our core")
print(f"  {n_intersection} are in the intersection (markers in BOTH lists)")
print(f"  {n_theirs_only} are theirs-only (in Hakkak's list, not in our core)")

## 7) What we learned

Three threads run through Sections 4–6 of this notebook: the headline recovery against Hakkak & Tohidfar 2026's published consensus, the regulatory-vs-effector split inside that headline, and the result of the pre-committed falsification test on canonical drought and salt markers. The next three subsections walk through each one. Section 7.4 writes the paper-ready artifacts to Drive, and Section 7.5 closes R1-Q1.

### 7.1) The headline

Hakkak & Tohidfar 2026 reported a 567-gene drought-and-salt-responsive consensus, built from a meta-analytic WGCNA on seven drought-stress datasets and six salinity datasets. Of those 567 AGI codes, 466 — about 82% — are also in our 8,138-gene core, derived from a completely independent pipeline: single-dataset DE on AtGenExpress's eight abiotic stresses, with the six-of-eight supermajority threshold from Notebook 3.

Two methodological gaps separate the analyses. Hakkak's consensus aggregates across thirteen drought-and-salinity studies; ours runs on one study (AtGenExpress) covering eight stresses. Their definition of "stress-responsive" is co-expression-network module membership; ours is DE-based with a cross-stress recurrence criterion. Despite both gaps, the gene-level convergence is strong: hypergeometric p ≈ 4.6e-111, about 2.2-fold enrichment over what random selection from the 21,249-gene background would produce.

The result reads two ways. From inside the field, an 82% recovery against an independent meta-analytic consensus tells us that the gene-level biology of plant abiotic stress response is reproducible across study design, dataset composition, and analytical method — not just within them. From outside, for an R1-Q1 single-dataset analysis to land near a published thirteen-dataset consensus is itself a reasonable validation that the analytical machinery built up in Notebooks 2 and 3 is doing what it claims to do.

### 7.2) Where it transfers and where it doesn't

Sections 5 and 6.1 separate the headline into two layers: regulatory genes (transcription factors and named PPI hubs) and effector genes (the things the regulators control). The asymmetry between the two layers is the most interpretively useful pattern in this comparison.

**Regulatory machinery transfers nearly perfectly.** Of Hakkak's 60 named transcription factors, 54 are in our core — 90% recovery, eight points above the headline. Eleven of their fifteen TF families are at 100% recovery, including the large families that do most of the stress-response work in the literature: ERF (9/9), WRKY (7/7), bHLH (7/7), C2H2 (4/4), bZIP (4/4), HD-ZIP (3/3), HSF (3/3). NAC sits at 8/9 (89%). MYB is the one consequential miss at 5/8 (62.5%) — worth a closer look in a follow-up but not enough on its own to disturb the regulatory pattern. The two missing singleton families (g2-like, and PHL1 listed under SBP) don't change the picture. Hakkak's 22 named hub genes show the same pattern: 19 in our core.

**Effector machinery is more variable.** The three missing named hubs from Section 6.1 — GASA9, PS2, MHX — are all non-TF: a putative gibberellin-regulated peptide, a phosphatase, and a magnesium-proton exchanger. They reach into specific physiology rather than into broadly-conserved stress regulation, which is consistent with them being more sensitive to study-design differences than the TFs are.

Two small caveats worth carrying into the paper's methods:

- AT2G47190 appears twice in Hakkak's TF supplementary file: once labeled "AtMYB2 / ERF" and once labeled "Atmyb2 / MYB" — the same AGI code with the family field disagreeing between rows. The ERF 9/9 figure above should really read 8/8 once the duplicate is collapsed. It doesn't change any qualitative claim, but it's worth correcting if we cite the TF count directly in the paper.
- DREB2A (AT5G05410), the principal hub in Hakkak's PPI network at degree 32, is not in their TF supplementary file 3, but it is in their main consensus file 1 and in our core. Their PPI analysis evidently ran on a slightly different gene set than the supplementary they reported the TF families from.

The compact reading: regulators of stress response (TFs and hubs) are the reproducible layer across methodologically very different analyses. Downstream specifics — which exchangers, which phosphatases, which small peptides — are where two pipelines on different datasets diverge.

### 7.3) The falsification anchor

Section 3 of this notebook committed to a pre-falsification prediction in writing: that canonical drought-and-salt-specific marker genes — RD29A, RD29B, COR47, ERD10, LEA14, the SOS pathway, NHX1, HKT1 — would land mostly in theirs-only, since they are the textbook signatures of the two stresses Hakkak studied and largely absent from the rest of what AtGenExpress covers. If the markers showed up in our core anyway, the meaning of "drought-and-salt-specific" itself would be called into question. That was the canonical hypothesis statement.

Section 6.2 ran the test on 10 such markers. Three results:

- **Four landed in the intersection** — RD29A, RD29B, ERD10, LEA14 — all classical drought / dehydration / LEA-family genes. Zero markers landed in theirs-only.
- **Three landed in core-only** — COR47, SOS3, HKT1 — meaning our 8-stress core flagged them as broadly stress-responsive even though Hakkak's drought-plus-salt consensus did not include them.
- **Three landed in neither region** — SOS1, SOS2, NHX1 — the canonical Na⁺ extrusion and compartmentation machinery, absent from both analyses.

The prediction failed in a specific, interpretable way. The drought-family markers (the four LEA / ERD / RD29 genes) are exactly where the textbook framing would not put them — inside the eight-stress consensus, not specific to drought and salt. The reading is not that the prediction was sloppy but that the textbook framing is sloppy: "drought-specific" markers are better described as "drought-prominent." Water-deprivation response overlaps at the gene level with osmotic, cold (which is partly dehydration), and heat (membrane integrity and proteostasis) responses. The canonical drought biology *is* broadly cross-stress.

This connects directly to the central finding of Notebook 2. Notebook 2's GO biological-process enrichment was hypoxia-dominant — not because the eight stresses are all hypoxic, but because they collectively converge on oxygen-status signaling that the drought-and-salt framing doesn't emphasize. Sections 4–6 of this notebook confirm the same pattern from the other direction: canonical drought/salt machinery, when present, isn't drought/salt-specific; it's part of a broader stress response. The two notebooks triangulate on the same conclusion using different evidence — one looking at what biological processes the 8-stress core enriches for, the other looking at what fraction of two specific stresses' published consensus the 8-stress core includes.

Two side notes worth carrying forward. The absence of SOS1, SOS2, and NHX1 from both analyses is interesting in itself: these are the canonical sodium-extrusion and compartmentation machinery, the molecular textbook for salt tolerance. Their absence from Hakkak's salt consensus suggests their stress-induced transcription may be smaller-amplitude than their post-translational regulation — something a transcriptomics meta-analysis would systematically miss. And HKT1 in core-only is intriguing — a Na⁺ transporter called stress-responsive in our analysis but not in Hakkak's. Worth checking on the AtGenExpress salt arms specifically before the paper, in case the signal is coming from cross-stress activation rather than from the salt comparison alone.

### 7.4) Save the paper-ready artifacts

Two files go to Drive. `comparison_summary.json` is the compact one — the pre-commitment prediction dict, the three-region counts, the hypergeometric statistics, the TF recovery numbers by family, the named-hub table, and the 10-marker falsification result. That's what the paper's main text will cite. `comparison_genes.parquet` is the gene-level supplementary — every AGI in our background annotated with which of the four regions (intersection / ours_only / theirs_only / neither) it falls into and whether it is one of Hakkak's named TFs. That's what the paper's supplementary tables get drawn from.

Both files write to `irilab2026_outputs/r1_q1/`. The save is followed by a round-trip read of each, with a shape check, to confirm the artifacts are usable.

In [ ]:
# Coerce to sets for set algebra and fast membership tests.
# (They may have arrived as lists from earlier sections.)
core_agi       = set(core_agi)
background_agi = set(background_agi)
hakkak_agi     = set(hakkak_agi)
intersection   = set(intersection)
theirs_only    = set(theirs_only)
ours_only      = set(ours_only)

import json
import pandas as pd

# --- Build the gene-level table (parquet artifact). ---

def _region(agi):
    in_core = agi in core_agi
    in_hakkak = agi in hakkak_agi
    if in_core and in_hakkak:
        return "intersection"
    if in_core:
        return "ours_only"
    if in_hakkak:
        return "theirs_only"
    return "neither"

# Hakkak's TF supplementary uses "ORF" as the AGI column.
tf_agis = set(hakkak_tf["ORF"].astype(str).str.upper().unique())

comparison_genes = pd.DataFrame({"agi": sorted(background_agi)})
comparison_genes["in_core"]      = comparison_genes["agi"].isin(core_agi)
comparison_genes["in_hakkak"]    = comparison_genes["agi"].isin(hakkak_agi)
comparison_genes["region"]       = comparison_genes["agi"].apply(_region)
comparison_genes["is_hakkak_tf"] = comparison_genes["agi"].isin(tf_agis)

# --- Build the summary dict (JSON artifact). ---

def _jsonable(obj):
    """Fallback serializer for sets and numpy scalars."""
    if isinstance(obj, set):
        return sorted(obj)
    if hasattr(obj, "item"):  # numpy scalar -> Python scalar
        return obj.item()
    return str(obj)

summary = {
    "comparison": "AtGenExpress 8-stress core vs Hakkak & Tohidfar 2026 drought-salt consensus",
    "background": {
        "N_total":                len(background_agi),
        "K_hakkak_in_background": len(hakkak_agi & background_agi),
        "n_core":                 len(core_agi),
        "k_intersection":         len(intersection),
    },
    "regions": {
        "intersection": len(intersection),
        "theirs_only":  len(theirs_only),
        "ours_only":    len(ours_only),
    },
    "hypergeometric": {
        "p_value":         float(p_value),
        "expected_k":      float(expected),
        "observed_k":      len(intersection),
        "fold_enrichment": len(intersection) / float(expected),
    },
    "recovery": {
        "hakkak_total":      len(hakkak_agi),
        "hakkak_in_core":    len(intersection),
        "percent_recovered": 100.0 * len(intersection) / len(hakkak_agi),
    },
    "prediction": prediction,
    "tf_recovery": {
        "total_tfs":   int(hakkak_tf["ORF"].nunique()),
        "tfs_in_core": int(hakkak_tf.loc[hakkak_tf["in_core"], "ORF"].nunique()),
        "by_family":   family_summary.to_dict(orient="records"),
    },
    "named_hubs": {
        "total":   int(len(named_df)),
        "in_core": int(named_df["in_core"].sum()),
        "table":   named_df.to_dict(orient="records"),
    },
    "falsification_markers": {
        "total":           int(len(markers_df)),
        "in_intersection": int((markers_df["in_core"] & markers_df["in_hakkak"]).sum()),
        "in_core_only":    int((markers_df["in_core"] & ~markers_df["in_hakkak"]).sum()),
        "in_theirs_only":  int((~markers_df["in_core"] & markers_df["in_hakkak"]).sum()),
        "in_neither":      int((~markers_df["in_core"] & ~markers_df["in_hakkak"] & markers_df["on_chip"]).sum()),
        "off_chip":        int((~markers_df["on_chip"]).sum()),
        "table":           markers_df.to_dict(orient="records"),
    },
}

# --- Write both files. ---

out_dir = iri.output_dir("r1_q1")

summary_path = out_dir / "comparison_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, default=_jsonable)
print(f"Saved: {summary_path}")
print(f"Size:  {summary_path.stat().st_size / 1024:.1f} KB")

genes_path = out_dir / "comparison_genes.parquet"
comparison_genes.to_parquet(genes_path)
print(f"Saved: {genes_path}")
print(f"Size:  {genes_path.stat().st_size / 1024:.1f} KB")
print(f"Shape: {comparison_genes.shape}")
print()
print("Region counts:")
print(comparison_genes["region"].value_counts())

# --- Round-trip both. ---

with open(summary_path) as f:
    summary_check = json.load(f)
print()
print(f"JSON round-trip: {len(summary_check)} top-level keys, "
      f"recovery {summary_check['recovery']['percent_recovered']:.1f}%")

genes_check = pd.read_parquet(genes_path)
assert genes_check.shape == comparison_genes.shape, \
    f"Parquet round-trip shape mismatch: {genes_check.shape} vs {comparison_genes.shape}"
print(f"Parquet round-trip: {genes_check.shape} OK")

### 7.5) Closing

That's R1-Q1 done analytically. Notebook 1 oriented to the question and pre-committed methodological choices; Notebook 2 ran differential expression across the 16 (stress, tissue) pairs of AtGenExpress; Notebook 3 distilled the cross-stress core and characterized its functional signature; and this notebook compared the core to Hakkak & Tohidfar 2026's drought-and-salt-specific consensus. The remaining R1-Q1 deliverables — the final paper and the presentation — are external to the notebook chain. They draw on artifacts written across all four notebooks, with `comparison_summary.json` and `comparison_genes.parquet` from Section 7.4 supplying the gene-list comparison content directly.